# Analysis of shape evolution during growth of propagules

## 1. Documentation and Imports


Created on 22-03-2021

Author: Valentin Laplaud

This code aims to characterize the shape of propagules growing in microfluidic chips. It uses the binarized image of the propagules generated by 'ColorPpglSizeTrackingNB.ipynb'

In [ ]:

# plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt

# numbers handling
import numpy as np
import pandas as pd

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes
import cv2 as cv

# to hide known warnings
import warnings

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf

## 2. Define analysis functions

## X. Enter data and run

# Test Zone

In [ ]:
from cycler import cycler

fig1 = plt.figure(dpi = 250)
mpl.rcParams['axes.prop_cycle'] = cycler(color=jet_colorcycle)
plt.plot(0,0,'*r')
fig2 = plt.figure(dpi = 250)
mpl.rcParams['axes.prop_cycle'] = cycler(color=jet_colorcycle)

n = 30

jet_colorcycle = [plt.get_cmap('jet')(1. * i/n) for i in range(n)]



for i in range(n):
    
    # Loading binary image
    BinImg = io.imread(r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.18\Processed\PPG2_Binarized\\' + str(i) + '.tif')

    # Find largest contour 
    cnts, _ = cv.findContours(BinImg, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
    cnt = max(cnts, key=cv.contourArea)

    # Create image with only largest object
    out = np.zeros(BinImg.shape, np.uint8)
    cv.drawContours(out, [cnt], -1, 255, cv.FILLED)
    BinImg = cv.bitwise_and(BinImg, out)

    # Detecting edges and computing their position
    edges = sobel(BinImg)>threshold_otsu(sobel(BinImg))
    indices = np.where(edges != [0])
    
    # Computing center of propagule using euclidian distance transform
    dist = cv.distanceTransform(BinImg, cv.DIST_L2, 3)
    regions = regionprops(BinImg,dist)
    center = regions[0].weighted_centroid
    
    relativeIndicesX = indices[0]-center[0]
    relativeIndicesY = indices[1]-center[1]
    
    Alpha = np.arctan2(relativeIndicesY,relativeIndicesX)

    Radius = np.sqrt(np.square(relativeIndicesX)+np.square(relativeIndicesY))

    FullCoordinates = np.stack((np.rad2deg(Alpha),Radius,relativeIndicesX,relativeIndicesY),axis=1)
    SortedCoord = FullCoordinates[FullCoordinates[:,0].argsort(),]
    
    
    plt.figure(fig1.number)
    plt.plot(SortedCoord[:,2],SortedCoord[:,3],'*', ms = 0.1)
    
    
    plt.figure(fig2.number)
    plt.plot(SortedCoord[:,1],'*', ms = 0.1)

plt.figure(fig1.number)
plt.gca().set_aspect('equal', adjustable='box')

plt.figure(dpi = 200)
plt.imshow(dist, cmap='gray')
plt.plot(indices[1],indices[0],'.b')
plt.plot(center[1],center[0],'*r')